# Akili Split CIFAR-100 — Official Mammoth Baseline Suite v1

Runs official Mammoth `er`, `derpp`, and `er_ace` baselines on `seq-cifar100`. It records the exact Mammoth commit, command line, logs, supported arguments, and generated artifacts for every seed.

This is baseline infrastructure only. It is not yet the final `DER++ + Akili validation/lifecycle` ablation.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount(os.getenv("AKILI_DERPP_DRIVE_MOUNT", "/content/drive"))
else:
    print("Not running in Colab; Drive mount skipped.")


In [ ]:
import ast, importlib, sys
from pathlib import Path

MODULE_NAME = "akili_mammoth_baseline_suite_v1"
MODULE_SOURCE = '\nfrom __future__ import annotations\n\nimport csv\nimport datetime as dt\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Mapping, Optional, Sequence, Set, Tuple\n\nPROTOCOL = "akili-mammoth-baseline-suite-v1"\nOFFICIAL_REPOSITORY = "https://github.com/aimagelab/mammoth.git"\n\n\ndef utc_now() -> str:\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef run_command(\n    command: Sequence[str],\n    *,\n    cwd: Path,\n    log_path: Path,\n    environment: Optional[Mapping[str, str]] = None,\n) -> int:\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ)\n    if environment:\n        env.update({key: str(value) for key, value in environment.items()})\n    with log_path.open("w", encoding="utf-8") as log:\n        process = subprocess.run(\n            list(command),\n            cwd=str(cwd),\n            env=env,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            check=False,\n        )\n    return int(process.returncode)\n\n\ndef git_commit(repo: Path) -> str:\n    process = subprocess.run(\n        ["git", "rev-parse", "HEAD"],\n        cwd=str(repo),\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    return process.stdout.strip() if process.returncode == 0 else "unknown"\n\n\ndef validate_mammoth(repo: Path) -> None:\n    required = [repo / "main.py", repo / "models", repo / "datasets", repo / "utils"]\n    missing = [str(path) for path in required if not path.exists()]\n    if missing:\n        raise FileNotFoundError(f"Invalid Mammoth repository: {missing}")\n\n\ndef discover_mammoth(project_root: Path, explicit: str = "") -> Path:\n    candidates: List[Path] = []\n    if explicit:\n        candidates.append(Path(explicit).expanduser())\n    candidates.extend([\n        project_root / "Mammoth",\n        project_root / "mammoth",\n        project_root / "third_party" / "mammoth",\n        Path("/content/mammoth"),\n    ])\n    for candidate in candidates:\n        if (candidate / "main.py").is_file():\n            validate_mammoth(candidate)\n            return candidate.resolve()\n    raise FileNotFoundError(\n        "Mammoth was not found. Set AKILI_MAMMOTH_ROOT to the existing Drive checkout. "\n        "The runner does not clone by default."\n    )\n\n\ndef cifar_integrity_root(candidate: Path) -> Optional[Path]:\n    candidate = candidate.expanduser().resolve()\n    possible = [candidate, candidate / "CIFAR100"]\n    for root in possible:\n        folder = root / "cifar-100-python"\n        if all((folder / name).is_file() for name in ("train", "test", "meta")):\n            return root.parent if root.name == "CIFAR100" else root\n    return None\n\n\ndef discover_cifar_base(project_root: Path, explicit: str = "") -> Path:\n    candidates: List[Path] = []\n    if explicit:\n        candidates.append(Path(explicit))\n    candidates.extend([\n        project_root / "data",\n        project_root / "datasets",\n        project_root,\n        Path("/content/drive/MyDrive/AKM_CLR/data"),\n    ])\n    for candidate in candidates:\n        if not candidate.exists():\n            continue\n        integrity = cifar_integrity_root(candidate)\n        if integrity is not None:\n            return integrity\n        for folder in candidate.rglob("cifar-100-python"):\n            if all((folder / name).is_file() for name in ("train", "test", "meta")):\n                if folder.parent.name == "CIFAR100":\n                    return folder.parent.parent.resolve()\n    raise FileNotFoundError(\n        "A complete CIFAR-100 cache was not found. This runner refuses to download it. "\n        "Set AKILI_DERPP_DATA_ROOT to the directory containing CIFAR100/cifar-100-python."\n    )\n\n\ndef help_flags(repo: Path, model: str, dataset: str) -> Set[str]:\n    process = subprocess.run(\n        [sys.executable, "main.py", "--model", model, "--dataset", dataset, "--help"],\n        cwd=str(repo),\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    text = process.stdout + "\\n" + process.stderr\n    return set(re.findall(r"--[A-Za-z0-9_-]+", text))\n\n\ndef add_if_supported(\n    command: List[str],\n    flags: Set[str],\n    name: str,\n    value: Optional[Any] = None,\n) -> None:\n    if name not in flags:\n        return\n    command.append(name)\n    if value is not None:\n        command.append(str(value))\n\n\ndef snapshot_files(repo: Path) -> Dict[str, Tuple[int, int]]:\n    roots = [repo / "results", repo / "logs", repo / "checkpoints"]\n    state: Dict[str, Tuple[int, int]] = {}\n    for root in roots:\n        if not root.exists():\n            continue\n        for path in root.rglob("*"):\n            if path.is_file():\n                stat = path.stat()\n                state[str(path.resolve())] = (stat.st_size, stat.st_mtime_ns)\n    return state\n\n\ndef copy_changed_files(\n    repo: Path,\n    before: Mapping[str, Tuple[int, int]],\n    destination: Path,\n) -> List[str]:\n    after = snapshot_files(repo)\n    changed = [Path(path) for path, sig in after.items() if before.get(path) != sig]\n    copied: List[str] = []\n    for source in changed:\n        relative = None\n        for root_name in ("results", "logs", "checkpoints"):\n            root = repo / root_name\n            try:\n                relative = Path(root_name) / source.relative_to(root.resolve())\n                break\n            except ValueError:\n                continue\n        if relative is None:\n            continue\n        target = destination / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source, target)\n        copied.append(relative.as_posix())\n    return copied\n\n\ndef extract_console_metrics(log_path: Path) -> Dict[str, Any]:\n    text = log_path.read_text(encoding="utf-8", errors="replace")\n    lines = [\n        line.strip()\n        for line in text.splitlines()\n        if any(token in line.lower() for token in (\n            "class-il", "task-il", "accuracy", "forgetting", "backward"\n        ))\n    ]\n    return {"matching_lines": lines[-100:]}\n\n\ndef build_command(\n    repo: Path,\n    *,\n    model: str,\n    dataset: str,\n    seed: int,\n    base_path: Path,\n    buffer_size: int,\n    minibatch_size: int,\n    mode: str,\n) -> Tuple[List[str], Set[str]]:\n    flags = help_flags(repo, model, dataset)\n    command = [sys.executable, "main.py", "--model", model, "--dataset", dataset]\n    add_if_supported(command, flags, "--seed", seed)\n    add_if_supported(command, flags, "--base_path", str(base_path))\n    add_if_supported(command, flags, "--num_workers", 0)\n    add_if_supported(command, flags, "--permute_classes", 0)\n    add_if_supported(command, flags, "--savecheck", "last")\n    add_if_supported(command, flags, "--csv_log", 1)\n    if mode == "smoke":\n        add_if_supported(command, flags, "--debug_mode", 1)\n    else:\n        add_if_supported(command, flags, "--model_config", "best")\n    if model in {"er", "der", "derpp", "er_ace"}:\n        add_if_supported(command, flags, "--buffer_size", buffer_size)\n        add_if_supported(command, flags, "--minibatch_size", minibatch_size)\n    return command, flags\n\n\ndef run_suite(\n    *,\n    repo: Path,\n    base_path: Path,\n    output_root: Path,\n    models: Sequence[str],\n    seeds: Sequence[int],\n    buffer_size: int,\n    minibatch_size: int,\n    mode: str,\n    force: bool = False,\n) -> Dict[str, Any]:\n    validate_mammoth(repo)\n    output_root.mkdir(parents=True, exist_ok=True)\n    commit = git_commit(repo)\n    records: List[Dict[str, Any]] = []\n\n    for model in models:\n        for seed in seeds:\n            run_root = output_root / model / f"seed_{seed}"\n            run_root.mkdir(parents=True, exist_ok=True)\n            complete_path = run_root / "COMPLETE.json"\n            if complete_path.is_file() and not force:\n                record = json.loads(complete_path.read_text(encoding="utf-8"))\n                if record.get("returncode") == 0:\n                    print(f"[resume] {model} seed={seed}")\n                    records.append(record)\n                    continue\n\n            command, flags = build_command(\n                repo,\n                model=model,\n                dataset="seq-cifar100",\n                seed=seed,\n                base_path=base_path,\n                buffer_size=buffer_size,\n                minibatch_size=minibatch_size,\n                mode=mode,\n            )\n            before = snapshot_files(repo)\n            log_path = run_root / "console.log"\n            print("[run]", " ".join(command))\n            returncode = run_command(\n                command,\n                cwd=repo,\n                log_path=log_path,\n                environment={"WANDB_MODE": "disabled", "PYTHONHASHSEED": str(seed)},\n            )\n            copied = copy_changed_files(repo, before, run_root / "mammoth_artifacts")\n            metrics = extract_console_metrics(log_path)\n            record = {\n                "protocol": PROTOCOL,\n                "model": model,\n                "dataset": "seq-cifar100",\n                "seed": seed,\n                "mode": mode,\n                "buffer_size": buffer_size if model in {"er", "der", "derpp", "er_ace"} else None,\n                "minibatch_size": minibatch_size if model in {"er", "der", "derpp", "er_ace"} else None,\n                "mammoth_commit": commit,\n                "command": command,\n                "supported_flags": sorted(flags),\n                "returncode": returncode,\n                "copied_artifacts": copied,\n                "console_metrics": metrics,\n                "completed_at": utc_now(),\n            }\n            complete_path.write_text(\n                json.dumps(record, indent=2, sort_keys=True), encoding="utf-8"\n            )\n            records.append(record)\n            if returncode != 0:\n                raise RuntimeError(\n                    f"{model} seed {seed} failed. Read {log_path}. "\n                    "Completed runs remain resumable."\n                )\n\n    summary = {\n        "protocol": PROTOCOL,\n        "created_at": utc_now(),\n        "mammoth_repository": str(repo),\n        "mammoth_commit": commit,\n        "cifar_base_path": str(base_path),\n        "models": list(models),\n        "seeds": list(seeds),\n        "mode": mode,\n        "records": records,\n        "all_completed": all(record["returncode"] == 0 for record in records),\n        "important_limit": (\n            "This suite measures official Mammoth baselines only. "\n            "It is not yet the Akili-vs-DER++ matched ablation."\n        ),\n    }\n    (output_root / "BASELINE_SUITE_SUMMARY.json").write_text(\n        json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8"\n    )\n    with (output_root / "BASELINE_SUITE_SUMMARY.csv").open(\n        "w", encoding="utf-8", newline=""\n    ) as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=[\n                "model", "seed", "mode", "buffer_size",\n                "minibatch_size", "mammoth_commit", "returncode",\n            ],\n        )\n        writer.writeheader()\n        for record in records:\n            writer.writerow({key: record.get(key) for key in writer.fieldnames})\n    return summary\n\n\ndef synthetic_verification(root: Path) -> Dict[str, Any]:\n    if root.exists():\n        shutil.rmtree(root)\n    repo = root / "mammoth"\n    (repo / "models").mkdir(parents=True)\n    (repo / "datasets").mkdir()\n    (repo / "utils").mkdir()\n    fake_main = (\n        "import sys\\n"\n        "if \'--help\' in sys.argv:\\n"\n        "    print(\'--seed --base_path --num_workers --permute_classes --savecheck "\n        "--csv_log --debug_mode --model_config --buffer_size --minibatch_size\')\\n"\n        "    raise SystemExit(0)\\n"\n        "print(\'Class-IL accuracy: 12.34\')\\n"\n    )\n    (repo / "main.py").write_text(fake_main, encoding="utf-8")\n    data = root / "data" / "CIFAR100" / "cifar-100-python"\n    data.mkdir(parents=True)\n    for name in ("train", "test", "meta"):\n        (data / name).write_text("x", encoding="utf-8")\n    base = discover_cifar_base(root, str(root / "data"))\n    command, flags = build_command(\n        repo,\n        model="derpp",\n        dataset="seq-cifar100",\n        seed=1,\n        base_path=base,\n        buffer_size=400,\n        minibatch_size=32,\n        mode="smoke",\n    )\n    checks = {\n        "repo_valid": True,\n        "cache_detected_without_download": base == (root / "data").resolve(),\n        "seed_in_command": "--seed" in command,\n        "base_path_in_command": "--base_path" in command,\n        "buffer_in_command": "--buffer_size" in command,\n        "debug_mode_in_smoke": "--debug_mode" in command,\n    }\n    return {"passed": all(checks.values()), "checks": checks, "command": command}\n'
runtime_root = Path("/tmp/akili_mammoth_suite")
runtime_root.mkdir(parents=True, exist_ok=True)
module_path = runtime_root / f"{MODULE_NAME}.py"
module_path.write_text(MODULE_SOURCE, encoding="utf-8")
ast.parse(MODULE_SOURCE)
compile(MODULE_SOURCE, str(module_path), "exec")
if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
suite = importlib.import_module(MODULE_NAME)
print("Protocol:", suite.PROTOCOL)


In [ ]:
import json, shutil
from pathlib import Path
test_root = Path("/tmp/akili_mammoth_suite_verify")
shutil.rmtree(test_root, ignore_errors=True)
verification = suite.synthetic_verification(test_root)
assert verification["passed"], verification
print(json.dumps(verification, indent=2))


In [ ]:
import os
from pathlib import Path
project_root = Path(os.getenv(
    "AKILI_DERPP_PROJECT_ROOT",
    "/content/drive/MyDrive/AKM_CLR",
))
mammoth_root = suite.discover_mammoth(
    project_root,
    os.getenv("AKILI_MAMMOTH_ROOT", ""),
)
cifar_base = suite.discover_cifar_base(
    project_root,
    os.getenv("AKILI_DERPP_DATA_ROOT", ""),
)
print("Mammoth:", mammoth_root)
print("Mammoth commit:", suite.git_commit(mammoth_root))
print("CIFAR base:", cifar_base)


In [ ]:
import os, subprocess, sys
if os.getenv("AKILI_DERPP_SKIP_INSTALL", "0").strip().lower() not in {
    "1", "true", "yes"
}:
    requirements = mammoth_root / "requirements.txt"
    if requirements.is_file():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
            check=True,
        )


In [ ]:
mode = os.getenv("AKILI_DERPP_MODE", "smoke").strip().lower()
if mode not in {"smoke", "publication"}:
    raise ValueError("AKILI_DERPP_MODE must be smoke or publication")
models = [
    value.strip() for value in os.getenv(
        "AKILI_DERPP_MODELS", "er,derpp,er_ace"
    ).split(",") if value.strip()
]
seeds = [
    int(value) for value in os.getenv(
        "AKILI_DERPP_SEEDS", "1" if mode == "smoke" else "1,2,3"
    ).split(",") if value.strip()
]
buffer_size = int(os.getenv("AKILI_DERPP_BUFFER_SIZE", "400"))
minibatch_size = int(os.getenv("AKILI_DERPP_MINIBATCH_SIZE", "32"))
force = os.getenv("AKILI_DERPP_FORCE", "0").strip().lower() in {
    "1", "true", "yes"
}
output_root = Path(os.getenv(
    "AKILI_DERPP_OUTPUT_ROOT",
    str(project_root / "stage06" / "mammoth_baseline_suite_v1"),
))
print({
    "mode": mode,
    "models": models,
    "seeds": seeds,
    "buffer_size": buffer_size,
    "minibatch_size": minibatch_size,
    "output_root": str(output_root),
})


In [ ]:
import json
SKIP_REAL = os.getenv("AKILI_DERPP_SKIP_REAL", "0").strip().lower() in {
    "1", "true", "yes"
}
if SKIP_REAL:
    RESULT = None
    print("Real Mammoth execution skipped.")
else:
    RESULT = suite.run_suite(
        repo=mammoth_root,
        base_path=cifar_base,
        output_root=output_root,
        models=models,
        seeds=seeds,
        buffer_size=buffer_size,
        minibatch_size=minibatch_size,
        mode=mode,
        force=force,
    )
    print(json.dumps({
        "all_completed": RESULT["all_completed"],
        "mammoth_commit": RESULT["mammoth_commit"],
        "records": len(RESULT["records"]),
        "output_root": str(output_root),
    }, indent=2))
